# LeetCode 2031 — Count Subarrays With More Ones Than Zeros bằng Python

Link đề: https://leetcode.com/problems/count-subarrays-with-more-ones-than-zeros/ · Độ khó: **Medium** · Chủ đề: Array, Binary Search, Divide and Conquer, Binary Indexed Tree, Segment Tree, Merge Sort, Ordered Set, Prefix Sum

> **Bài Premium:** đề trong notebook được viết lại theo hiểu biết về bài (nguồn doocs). Nếu có tài khoản Premium, nên đối chiếu lại câu chữ và ví dụ với đề gốc. Lưu ý: tên method trên LeetCode (theo doocs) là `subarraysWithMoreZerosThanOnes` dù đề đếm đoạn có **nhiều số 1 hơn**.

Skeleton LeetCode:

```python
class Solution:
    def subarraysWithMoreZerosThanOnes(self, nums: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

1. Biến bài "đếm đoạn con có nhiều số 1 hơn số 0" thành bài **đếm cặp tiền tố** `P[i] < P[j]` bằng phép đổi `0 → −1`.
2. Đếm cặp `i < j` với `P[i] < P[j]` bằng **sắp xếp trộn** (merge sort — họ hàng của đếm nghịch thế) trong `O(n log n)`.
3. Dùng **cây Fenwick (BIT)** với **dịch chỉ số** (offset) để xử lý tiền tố âm, `O(n log n)`.
4. Khai thác tính chất "tiền tố chỉ thay đổi ±1 mỗi bước" để có lời giải **tuyến tính `O(n)`**.
5. Xử lý modulo `10⁹ + 7`, và kiểm thử đối chiếu brute force trên mọi mảng nhị phân ngắn và dữ liệu ngẫu nhiên.

## 2. Đề bài & ví dụ

Cho mảng nhị phân `nums` chỉ gồm `0` và `1`. Đếm số **đoạn con liên tiếp** có **số chữ 1 nhiều hơn số chữ 0**. Đáp án có thể rất lớn ⇒ trả về modulo `10⁹ + 7`.

**Ví dụ 1:** `nums = [0,1,1,0,1]` → `9`:
- dài 1: `[1]`, `[1]`, `[1]`;
- dài 2: `[1,1]`;
- dài 3: `[0,1,1]`, `[1,1,0]`, `[1,0,1]`;
- dài 4: `[1,1,0,1]`;
- dài 5: `[0,1,1,0,1]`.

**Ví dụ 2:** `[0]` → `0`. **Ví dụ 3:** `[1]` → `1`.

**Ràng buộc:** `1 ≤ n ≤ 10⁵`, `nums[i] ∈ {0, 1}`.

In [1]:
import random
import time
from bisect import bisect_left, insort
from itertools import accumulate, product
from typing import List

MOD = 10**9 + 7
examples = [([0, 1, 1, 0, 1], 9), ([0], 0), ([1], 1)]
for nums, exp in examples:
    print(nums, "-> kỳ vọng", exp)

[0, 1, 1, 0, 1] -> kỳ vọng 9
[0] -> kỳ vọng 0
[1] -> kỳ vọng 1


## 3. Kiến thức nền

### 3.1 Đổi `0 → −1`: "nhiều 1 hơn 0" ⇔ "tổng dương"

Thay mỗi `0` bằng `−1`, mỗi `1` giữ nguyên. Một đoạn có `a` số 1 và `b` số 0 có tổng `a − b`; nhiều 1 hơn ⇔ tổng `> 0`.

In [2]:
nums = [0, 1, 1, 0, 1]
v = [1 if x else -1 for x in nums]
print("v =", v)
print("đoạn [1,0,1] -> tổng", sum([1, -1, 1]), "> 0: nhiều 1 hơn")

v = [-1, 1, 1, -1, 1]
đoạn [1,0,1] -> tổng 1 > 0: nhiều 1 hơn


### 3.2 Mảng tiền tố và tổng đoạn

`P[0] = 0`, `P[k] = v[0] + ... + v[k−1]`. Tổng đoạn `v[i..j−1] = P[j] − P[i]`. Vậy đoạn `[i, j)` hợp lệ ⇔ `P[i] < P[j]` (với `i < j`).

⇒ **Đáp án = số cặp `0 ≤ i < j ≤ n` có `P[i] < P[j]`.** Đừng quên `P[0] = 0` (đoạn bắt đầu từ chỉ số 0).

In [3]:
P = [0] + list(accumulate(v))
print("P =", P)
pairs = [(i, j) for i in range(len(P)) for j in range(i + 1, len(P)) if P[i] < P[j]]
print("số cặp P[i] < P[j]:", len(pairs))

P = [0, -1, 0, 1, 0, 1]
số cặp P[i] < P[j]: 9


### 3.3 Cây Fenwick (Binary Indexed Tree)

Hỗ trợ hai thao tác `O(log N)` trên mảng đếm `cnt[1..N]`: `update(x, +1)` và `query(x) = cnt[1] + ... + cnt[x]` (tổng tiền tố). `x & -x` là bit 1 thấp nhất của `x`. **Chỉ số phải ≥ 1** — với giá trị âm cần **dịch** (cộng offset).

In [4]:
class BIT:
    def __init__(self, n):
        self.n = n
        self.t = [0] * (n + 1)
    def update(self, x, d):
        while x <= self.n:
            self.t[x] += d
            x += x & -x
    def query(self, x):            # tổng cnt[1..x]
        s = 0
        while x > 0:
            s += self.t[x]
            x -= x & -x
        return s

b = BIT(10)
for x in [3, 5, 5, 9]:
    b.update(x, 1)
print("số phần tử <= 5:", b.query(5), "| <= 4:", b.query(4), "| <= 10:", b.query(10))

số phần tử <= 5: 3 | <= 4: 1 | <= 10: 4


### 3.4 Đếm cặp `i < j`, `a[i] < a[j]` bằng merge sort

Khi trộn hai nửa đã sắp `L` và `R` (mọi phần tử của `L` đứng **trước** `R` trong mảng gốc), với mỗi `y ∈ R`, số `x ∈ L` có `x < y` là số phần tử của `L` nhỏ hơn `y` — đếm bằng con trỏ trong lúc trộn. Đây là phiên bản "ngược" của đếm nghịch thế.

In [5]:
def count_less_pairs(a):
    if len(a) <= 1:
        return a[:], 0
    mid = len(a) // 2
    L, cl = count_less_pairs(a[:mid]); R, cr = count_less_pairs(a[mid:])
    cnt, i = 0, 0
    for y in R:                      # L, R đã sắp tăng
        while i < len(L) and L[i] < y:
            i += 1
        cnt += i                     # L[0..i-1] < y
    return sorted(L + R), cl + cr + cnt

print(count_less_pairs([0, -1, 0, 1, 0, 1])[1], "(so với đếm trực tiếp:", len(pairs), ")")

9 (so với đếm trực tiếp: 9 )


### 3.5 Bước đi ±1 của tiền tố

Vì mỗi `v[k] ∈ {−1, +1}`, `P[k+1] = P[k] ± 1` — tiền tố là một **đường đi ngẫu nhiên** trên các số nguyên, giá trị nằm trong `[−n, n]`. Tính chất "mỗi bước chỉ nhích 1" là chìa khóa cho lời giải `O(n)`.

In [6]:
print("P:", P, "| các bước:", [P[k + 1] - P[k] for k in range(len(P) - 1)])

P: [0, -1, 0, 1, 0, 1] | các bước: [-1, 1, 1, -1, 1]


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 1: `P = [0, −1, 0, 1, 0, 1]`. Với mỗi `j`, đếm số `i < j` có `P[i] < P[j]`:

| j | P[j] | các P[i] trước | số P[i] < P[j] |
|---|---|---|---|
| 1 | −1 | 0 | 0 |
| 2 | 0 | 0, −1 | 1 |
| 3 | 1 | 0, −1, 0 | 3 |
| 4 | 0 | 0, −1, 0, 1 | 1 |
| 5 | 1 | 0, −1, 0, 1, 0 | 4 |

Tổng `0 + 1 + 3 + 1 + 4 = 9` ✓. Bài toán trở thành: duyệt `j` và hỏi "trong các tiền tố trước, có bao nhiêu cái **nhỏ hơn** tiền tố hiện tại?".

In [7]:
total = 0
for j in range(1, len(P)):
    c = sum(1 for i in range(j) if P[i] < P[j])
    total += c
    print(f"j={j}: P[j]={P[j]:2d}, số P[i] nhỏ hơn = {c}, cộng dồn = {total}")

j=1: P[j]=-1, số P[i] nhỏ hơn = 0, cộng dồn = 0
j=2: P[j]= 0, số P[i] nhỏ hơn = 1, cộng dồn = 1
j=3: P[j]= 1, số P[i] nhỏ hơn = 3, cộng dồn = 4
j=4: P[j]= 0, số P[i] nhỏ hơn = 1, cộng dồn = 5
j=5: P[j]= 1, số P[i] nhỏ hơn = 4, cộng dồn = 9


## 5. Cách 1 — Brute Force / baseline: mọi đoạn con, cộng dồn

**Ý tưởng:** cố định đầu `i`, kéo dài `j` và duy trì `ones − zeros`; mỗi khi `> 0` thì đếm.

In [8]:
def count_brute(nums):
    n = len(nums); ans = 0
    for i in range(n):
        bal = 0
        for j in range(i, n):
            bal += 1 if nums[j] else -1
            if bal > 0:
                ans += 1
    return ans % MOD

for nums, exp in examples:
    got = count_brute(nums); print(nums, "->", got); assert got == exp

[0, 1, 1, 0, 1] -> 9
[0] -> 0
[1] -> 1


**Độ phức tạp:** `O(n²)` — `n = 10⁵` ⇒ `5·10⁹` bước, quá chậm. Space `O(1)`.

**Nút thắt:** với mỗi `j`, ta "hỏi lại" mọi `i`. Cần một cấu trúc trả lời nhanh "có bao nhiêu tiền tố trước nhỏ hơn `x`".

In [9]:
rng = random.Random(0)
for n in (1000, 2000, 4000):
    a = [rng.randint(0, 1) for _ in range(n)]
    t = time.perf_counter(); count_brute(a)
    print(f"n={n}: {time.perf_counter() - t:.3f}s")

n=1000: 0.027s
n=2000: 0.084s


n=4000: 0.336s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (quy về đếm cặp):** đáp án `= #{(i, j) : 0 ≤ i < j ≤ n, P[i] < P[j]}`.

**Insight 2 (cấu trúc đếm):** duyệt `j` tăng dần, giữ **đa tập** các `P[0..j−1]`; hỏi số phần tử `< P[j]`; rồi thêm `P[j]`. Dùng BIT (giá trị trong `[−n, n]` ⇒ dịch `+ n + 1`) hoặc chia để trị.

**Insight 3 (tuyến tính):** gọi `less` = số tiền tố đã thấy **nhỏ hơn** giá trị hiện tại `s`. Khi bước sang `s' = s + 1`: các tiền tố **bằng** `s` trở thành nhỏ hơn ⇒ `less += cnt[s]`. Khi bước sang `s' = s − 1`: các tiền tố bằng `s − 1` không còn nhỏ hơn ⇒ `less −= cnt[s − 1]`. Mỗi bước `O(1)`!

**Quy tắc "nếu… thì…" cho cách tuyến tính** (trước khi thêm `P[j−1]` vào… cẩn thận thứ tự — xem code):
- **Nếu** `v = +1` **thì** `less += cnt[s]` rồi `s += 1`.
- **Nếu** `v = −1` **thì** `s −= 1` rồi `less −= cnt[s]`.
- Cộng `less` vào đáp án, rồi `cnt[s] += 1`.

**Invariant:** trước khi cộng, `less = #{i < j : P[i] < P[j]}` và `cnt[x] = #{i < j : P[i] = x}`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Tiền tố + sắp xếp trộn (chia để trị) `O(n log n)`

Đếm cặp `i < j`, `P[i] < P[j]` trên mảng `P` (độ dài `n + 1`) bằng hàm `count_less_pairs` ở mục 3.4. Ở đây viết bản lặp hiệu quả hơn (không tạo nhiều list con).

In [10]:
def count_mergesort(nums):
    P = [0]
    for x in nums:
        P.append(P[-1] + (1 if x else -1))
    def solve(lo, hi):                      # đếm trên P[lo:hi], đồng thời sắp đoạn đó
        if hi - lo <= 1:
            return 0
        mid = (lo + hi) // 2
        cnt = solve(lo, mid) + solve(mid, hi)
        i = lo
        for k in range(mid, hi):            # hai nửa đã sắp tăng
            y = P[k]
            while i < mid and P[i] < y:
                i += 1
            cnt += i - lo
        P[lo:hi] = sorted(P[lo:hi])         # trộn (dùng sorted cho gọn; vẫn O(n log n) tổng thể ~)
        return cnt
    return solve(0, len(P)) % MOD

for nums, exp in examples:
    assert count_mergesort(nums) == exp
print([count_mergesort(nums) for nums, _ in examples])

[9, 0, 1]


**Độ phức tạp Cách 2:** mỗi tầng đệ quy duyệt `O(n)` phần tử, `log n` tầng ⇒ `O(n log n)` (dùng `sorted` trên hai nửa đã sắp, Timsort nhận ra hai "run" nên trộn gần tuyến tính). Độ sâu đệ quy chỉ `log n` ≈ 17. Space `O(n)`.

### 7.2 Cách 3 — Tiền tố + cây Fenwick với dịch chỉ số

**Giải thích từng đoạn:**
1. Tiền tố `s ∈ [−n, n]` ⇒ chỉ số BIT `s + n + 1 ∈ [1, 2n+1]`.
2. Ban đầu thêm `P[0] = 0`.
3. Với mỗi phần tử: cập nhật `s`; `ans += query(s − 1 + off)` (số tiền tố `≤ s − 1`, tức `< s`); thêm `s`.

In [11]:
def count_bit(nums):
    n = len(nums); off = n + 1
    bit = BIT(2 * n + 1)
    bit.update(0 + off, 1)                  # P[0] = 0
    s = ans = 0
    for x in nums:
        s += 1 if x else -1
        ans += bit.query(s - 1 + off)       # số tiền tố trước < s
        bit.update(s + off, 1)
    return ans % MOD

for nums, exp in examples:
    assert count_bit(nums) == exp
print([count_bit(nums) for nums, _ in examples])

[9, 0, 1]


In [12]:
class SolutionBIT:
    def subarraysWithMoreZerosThanOnes(self, nums: List[int]) -> int:
        n = len(nums); off = n + 1; size = 2 * n + 1
        tree = [0] * (size + 1)
        def update(x):
            while x <= size:
                tree[x] += 1; x += x & -x
        def query(x):
            r = 0
            while x > 0:
                r += tree[x]; x -= x & -x
            return r
        update(off)
        s = ans = 0
        for x in nums:
            s += 1 if x else -1
            ans += query(s - 1 + off)
            update(s + off)
        return ans % (10**9 + 7)

print([SolutionBIT().subarraysWithMoreZerosThanOnes(nums) for nums, _ in examples])

[9, 0, 1]


**Độ phức tạp Cách 3:** `O(n log n)` thời gian, `O(n)` bộ nhớ.

### 7.3 Cách 4 — Danh sách đã sắp + `bisect` (đơn giản, `O(n²)` worst nhưng nhanh trong thực tế)

Giữ list `seen` đã sắp các tiền tố; `bisect_left(seen, s)` = số phần tử `< s`; `insort` để chèn. `insort` dịch phần tử `O(n)` nhưng bằng `memmove` trong C nên vẫn nhanh với `n = 10⁵`.

In [13]:
def count_bisect(nums):
    seen = [0]; s = ans = 0
    for x in nums:
        s += 1 if x else -1
        ans += bisect_left(seen, s)
        insort(seen, s)
    return ans % MOD

for nums, exp in examples:
    assert count_bisect(nums) == exp
print([count_bisect(nums) for nums, _ in examples])

[9, 0, 1]


**Độ phức tạp Cách 4:** `O(n log n)` cho tìm kiếm + `O(n²)` worst cho dịch phần tử khi chèn (hằng số rất nhỏ). Space `O(n)`.

### 7.4 Cách 5 — Tuyến tính nhờ bước ±1 (bản nộp chính)

**Giải thích từng đoạn:**
1. `cnt` là mảng đếm với offset `n` (giá trị `s` lưu ở `cnt[s + n]`).
2. `less` = số tiền tố đã thấy `< s` (giá trị hiện tại).
3. Gặp `1`: các tiền tố bằng `s` (cũ) trở thành nhỏ hơn `s + 1` ⇒ `less += cnt[s]`; `s += 1`.
4. Gặp `0`: `s −= 1`; các tiền tố bằng `s` (mới) không còn nhỏ hơn ⇒ `less −= cnt[s]`.
5. `ans += less`; `cnt[s] += 1`.

In [14]:
def count_linear(nums):
    n = len(nums)
    cnt = [0] * (2 * n + 1)
    off = n
    cnt[0 + off] = 1                     # P[0] = 0
    s = less = ans = 0
    for x in nums:
        if x == 1:
            less += cnt[s + off]         # các tiền tố bằng s cũ giờ < s mới
            s += 1
        else:
            s -= 1
            less -= cnt[s + off]         # các tiền tố bằng s mới không còn < s
        ans += less
        cnt[s + off] += 1
    return ans % MOD

for nums, exp in examples:
    assert count_linear(nums) == exp
print([count_linear(nums) for nums, _ in examples])

[9, 0, 1]


**Độ phức tạp Cách 5:** `O(n)` thời gian, `O(n)` bộ nhớ (mảng `cnt`).

## 8. TRACE từng bước

Trace Cách 5 trên ví dụ 1 `[0,1,1,0,1]`. Ban đầu `s = 0`, `cnt = {0: 1}`, `less = 0`.

| j | x | thao tác | s | less | ans | cnt sau |
|---|---|---|---|---|---|---|
| 1 | 0 | s=−1, less −= cnt[−1]=0 | −1 | 0 | 0 | {0:1, −1:1} |
| 2 | 1 | less += cnt[−1]=1, s=0 | 0 | 1 | 1 | {0:2, −1:1} |
| 3 | 1 | less += cnt[0]=2, s=1 | 1 | 3 | 4 | {0:2, −1:1, 1:1} |
| 4 | 0 | s=0, less −= cnt[0]=2 | 0 | 1 | 5 | {0:3, −1:1, 1:1} |
| 5 | 1 | less += cnt[0]=3, s=1 | 1 | 4 | 9 | {..., 1:2} |

In [15]:
def trace_linear(nums):
    from collections import defaultdict
    cnt = defaultdict(int); cnt[0] = 1
    s = less = ans = 0
    for j, x in enumerate(nums, 1):
        if x == 1:
            add = cnt[s]; less += add; s += 1; op = f"less += cnt[{s - 1}]={add}, s -> {s}"
        else:
            s -= 1; sub = cnt[s]; less -= sub; op = f"s -> {s}, less -= cnt[{s}]={sub}"
        ans += less; cnt[s] += 1
        print(f"j={j} x={x}: {op:28s} | less={less} ans={ans} cnt={dict(sorted(cnt.items()))}")
    return ans

print("Đáp án:", trace_linear([0, 1, 1, 0, 1]))

j=1 x=0: s -> -1, less -= cnt[-1]=0   | less=0 ans=0 cnt={-1: 1, 0: 1}
j=2 x=1: less += cnt[-1]=1, s -> 0    | less=1 ans=1 cnt={-1: 1, 0: 2}
j=3 x=1: less += cnt[0]=2, s -> 1     | less=3 ans=4 cnt={-1: 1, 0: 2, 1: 1}
j=4 x=0: s -> 0, less -= cnt[0]=2     | less=1 ans=5 cnt={-1: 1, 0: 3, 1: 1}
j=5 x=1: less += cnt[0]=3, s -> 1     | less=4 ans=9 cnt={-1: 1, 0: 3, 1: 2}
Đáp án: 9


In [16]:
def trace_bit(nums):
    n = len(nums); off = n + 1; bit = BIT(2 * n + 1); bit.update(off, 1); s = ans = 0
    for j, x in enumerate(nums, 1):
        s += 1 if x else -1
        q = bit.query(s - 1 + off); ans += q; bit.update(s + off, 1)
        print(f"j={j}: s={s:2d}, chỉ số BIT={s + off}, số tiền tố < s = {q}, ans={ans}")
    return ans

print("Đáp án:", trace_bit([1, 1, 0, 0, 1]))

j=1: s= 1, chỉ số BIT=7, số tiền tố < s = 1, ans=1
j=2: s= 2, chỉ số BIT=8, số tiền tố < s = 2, ans=3
j=3: s= 1, chỉ số BIT=7, số tiền tố < s = 1, ans=4
j=4: s= 0, chỉ số BIT=6, số tiền tố < s = 0, ans=4
j=5: s= 1, chỉ số BIT=7, số tiền tố < s = 2, ans=6
Đáp án: 6


## 9. Code nộp LeetCode

In [17]:
class Solution:
    def subarraysWithMoreZerosThanOnes(self, nums: List[int]) -> int:
        MOD = 10**9 + 7
        n = len(nums)
        cnt = [0] * (2 * n + 1)          # cnt[s + n] = số tiền tố đã thấy có giá trị s
        cnt[n] = 1                       # tiền tố rỗng P[0] = 0
        s = less = ans = 0               # less = số tiền tố đã thấy < s
        for x in nums:
            if x == 1:
                less += cnt[s + n]
                s += 1
            else:
                s -= 1
                less -= cnt[s + n]
            ans += less
            cnt[s + n] += 1
        return ans % MOD

for nums, exp in examples:
    assert Solution().subarraysWithMoreZerosThanOnes(nums) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Quy về đếm cặp.** Đoạn `nums[i..j−1]` (`0 ≤ i < j ≤ n`) có `ones − zeros = P[j] − P[i]`; hợp lệ ⇔ `P[i] < P[j]`. Mỗi đoạn ↔ đúng một cặp `(i, j)`.

**Bất biến của Cách 5.** Đặt `S_j = {P[0], ..., P[j−1]}` (đa tập). Ta chứng minh trước khi cộng ở bước `j`: `less = #{y ∈ S_j : y < P[j]}`.
- Trước bước 1, `S = {0}`, `s = 0`, `less = 0` = số phần tử `< 0` ✓.
- Giả sử sau bước `j − 1` (đã thêm `P[j−1]` vào `cnt`), `less = #{y ∈ S_j : y < P[j−1]}` — vì phần tử vừa thêm bằng `P[j−1]` không nhỏ hơn chính nó nên giá trị `less` không đổi khi thêm.
- Nếu `P[j] = P[j−1] + 1`: `#{y < P[j]} = #{y < P[j−1]} + #{y = P[j−1]}` ⇒ `less += cnt[P[j−1]]` ✓.
- Nếu `P[j] = P[j−1] − 1`: `#{y < P[j]} = #{y < P[j−1]} − #{y = P[j]}` ⇒ `less −= cnt[P[j]]` ✓.

Vậy `ans = Σ_j #{i < j : P[i] < P[j]}` đúng bằng số cặp. Modulo lấy ở cuối là đủ vì Python không tràn số (đáp án ≤ `n(n+1)/2`).

## 11. Phân tích Time / Space

Gọi `n = len(nums)`.

| Cách | Time | Space |
|---|---|---|
| 1. Brute force | `O(n²)` | `O(1)` |
| 2. Merge sort | `O(n log n)` | `O(n)` |
| 3. Fenwick | `O(n log n)` | `O(n)` |
| 4. `bisect` + `insort` | `O(n log n)` tìm + `O(n²)` dịch (hằng số nhỏ) | `O(n)` |
| 5. Tuyến tính ±1 | `O(n)` | `O(n)` |

Đáp án lớn nhất khi toàn số 1: `n(n+1)/2 ≈ 5·10⁹ > 10⁹ + 7` ⇒ modulo có tác dụng thật.

In [18]:
rng = random.Random(2031)
big = [rng.randint(0, 1) for _ in range(10**5)]
ones = [1] * 10**5
for name, fn in [("mergesort", count_mergesort), ("bit", count_bit), ("bisect", count_bisect),
                 ("linear", count_linear), ("Solution", lambda a: Solution().subarraysWithMoreZerosThanOnes(a))]:
    t = time.perf_counter(); r1 = fn(big); r2 = fn(ones)
    print(f"{name:9s}: ngẫu nhiên={r1}, toàn 1={r2} ({time.perf_counter() - t:.3f}s)")
print("n(n+1)/2 =", 10**5 * (10**5 + 1) // 2, "-> mod =", (10**5 * (10**5 + 1) // 2) % MOD)

mergesort: ngẫu nhiên=949458865, toàn 1=49965 (0.227s)


bit      : ngẫu nhiên=949458865, toàn 1=49965 (0.245s)


bisect   : ngẫu nhiên=949458865, toàn 1=49965 (0.496s)
linear   : ngẫu nhiên=949458865, toàn 1=49965 (0.019s)
Solution : ngẫu nhiên=949458865, toàn 1=49965 (0.019s)
n(n+1)/2 = 5000050000 -> mod = 49965


## 12. Bộ test

- Ví dụ đề.
- **Vét cạn mọi mảng nhị phân** độ dài `1..12` (≈ 8000 mảng), so mọi cách với brute force.
- Biên: toàn 0 (đáp án 0), toàn 1 (`n(n+1)/2`), xen kẽ.
- Ngẫu nhiên dài hơn (seed cố định), và kiểm modulo trên mảng toàn 1 lớn.

In [19]:
fns = {
    "brute": count_brute, "mergesort": count_mergesort, "bit": count_bit, "bisect": count_bisect,
    "linear": count_linear, "Solution": lambda a: Solution().subarraysWithMoreZerosThanOnes(a),
    "SolutionBIT": lambda a: SolutionBIT().subarraysWithMoreZerosThanOnes(a),
}
for nums, exp in examples:
    for name, fn in fns.items():
        assert fn(nums) == exp, (name, nums)
total = 0
for n in range(1, 13):
    for tup in product((0, 1), repeat=n):
        a = list(tup); want = count_brute(a)
        for name, fn in fns.items():
            assert fn(a) == want, (name, a)
        total += 1
for n in (1, 2, 7, 50):
    assert Solution().subarraysWithMoreZerosThanOnes([0] * n) == 0
    assert Solution().subarraysWithMoreZerosThanOnes([1] * n) == n * (n + 1) // 2
print(f"Ví dụ + vét cạn {total} mảng (n <= 12) + biên: OK")

Ví dụ + vét cạn 8190 mảng (n <= 12) + biên: OK


In [20]:
rng = random.Random(42)
for _ in range(200):
    n = rng.randint(1, 300); p = rng.random()
    a = [1 if rng.random() < p else 0 for _ in range(n)]
    want = count_brute(a)
    for name, fn in fns.items():
        assert fn(a) == want, (name, a)
big1 = [1] * 70000
assert count_linear(big1) == count_bit(big1) == (70000 * 70001 // 2) % MOD
print("200 test ngẫu nhiên + kiểm modulo trên 70000 số 1: OK")

200 test ngẫu nhiên + kiểm modulo trên 70000 số 1: OK


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Đếm `P[i] ≤ P[j]` (tính cả đoạn có số 1 **bằng** số 0)

In [21]:
from bisect import bisect_right
def wrong_le(nums):
    seen = [0]; s = ans = 0
    for x in nums:
        s += 1 if x else -1
        ans += bisect_right(seen, s)                 # SAI: đếm cả P[i] == P[j]
        insort(seen, s)
    return ans % MOD

nums, exp = examples[0]
print("sai:", wrong_le(nums), "| đúng:", exp)

sai: 13 | đúng: 9


In [22]:
print("sửa (bisect_left: chỉ đếm P[i] < s):", count_bisect(nums))

sửa (bisect_left: chỉ đếm P[i] < s): 9


### Lỗi 2 — Quên tiền tố rỗng `P[0] = 0`

Thiếu nó thì bỏ sót mọi đoạn **bắt đầu tại chỉ số 0** (ví dụ 3 `[1]` sẽ ra 0).

In [23]:
def wrong_no_empty_prefix(nums):
    seen = []; s = ans = 0                           # SAI: không có P[0] = 0
    for x in nums:
        s += 1 if x else -1
        ans += bisect_left(seen, s); insort(seen, s)
    return ans % MOD

print("sai:", [wrong_no_empty_prefix(a) for a, _ in examples], "| đúng:", [e for _, e in examples])

sai: [7, 0, 0] | đúng: [9, 0, 1]


In [24]:
print("sửa:", [count_bisect(a) for a, _ in examples])

sửa: [9, 0, 1]


### Lỗi 3 — Sai thứ tự cập nhật khi gặp số 0 (trừ `cnt` của giá trị **cũ**)

Khi đi xuống `s → s − 1`, các tiền tố **bằng `s − 1`** (giá trị mới) mới là những cái "không còn nhỏ hơn". Trừ `cnt[s]` (giá trị cũ) trước khi giảm `s` là sai.

*Ghi chú bên lề về offset:* nếu dùng mảng `cnt` cỡ `n + 1` và chỉ số âm, Python "quấn" `cnt[-k]` về `cnt[n + 1 − k]`; may mắn là trong `n` bước tiền tố không thể vừa chạm `−k` vừa chạm `n + 1 − k` nên **tình cờ** vẫn đúng — nhưng đừng dựa vào may mắn đó (ngôn ngữ khác sẽ lỗi truy cập, còn BIT với chỉ số ≤ 0 sẽ lặp vô hạn vì `0 & -0 = 0`). Luôn dùng offset rõ ràng.

In [25]:
def wrong_order(nums):
    n = len(nums); cnt = [0] * (2 * n + 1); cnt[n] = 1
    s = less = ans = 0
    for x in nums:
        if x == 1:
            less += cnt[s + n]; s += 1
        else:
            less -= cnt[s + n]; s -= 1               # SAI: phải giảm s trước rồi mới trừ cnt[s]
        ans += less; cnt[s + n] += 1
    return ans % MOD

a = next(list(t) for n in range(1, 10) for t in product((0, 1), repeat=n)
         if wrong_order(list(t)) != count_brute(list(t)))
print("phản ví dụ ngắn nhất:", a, "| sai:", wrong_order(a), "| đúng:", count_brute(a))

phản ví dụ ngắn nhất: [0] | sai: 1000000006 | đúng: 0


In [26]:
print("sửa:", count_linear(a), "| ví dụ 1:", wrong_order([0, 1, 1, 0, 1]), "(sai) vs", count_linear([0, 1, 1, 0, 1]))

sửa: 0 | ví dụ 1: 6 (sai) vs 9


### Lỗi 4 — Quên lấy modulo

Python không tràn số nên đáp án "đúng toán học" nhưng **sai theo đề** khi vượt `10⁹ + 7`.

In [27]:
def wrong_no_mod(nums):
    n = len(nums); cnt = [0] * (2 * n + 1); cnt[n] = 1; s = less = ans = 0
    for x in nums:
        if x: less += cnt[s + n]; s += 1
        else: s -= 1; less -= cnt[s + n]
        ans += less; cnt[s + n] += 1
    return ans                                       # SAI: thiếu % MOD

a = [1] * 50000
print("sai:", wrong_no_mod(a), "| đúng:", Solution().subarraysWithMoreZerosThanOnes(a))

sai: 1250025000 | đúng: 250024993


In [28]:
print("sửa:", wrong_no_mod(a) % MOD)

sửa: 250024993


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Đếm đoạn con có (số loại A) > (số loại B)"** ⇒ đổi thành `±1`, quy về đếm cặp tiền tố `P[i] < P[j]`.
- **Đếm cặp `i < j` có `a[i] < a[j]`** ⇒ BIT (nén/dịch chỉ số), merge sort, hoặc cấu trúc có thứ tự.
- **Giá trị thay đổi ±1 mỗi bước** ⇒ thường có lời giải `O(n)` bằng cập nhật "bộ đếm nhỏ hơn" dần dần.
- Tiền tố có thể âm ⇒ **offset**.
- Luôn nhớ tiền tố rỗng `P[0] = 0`.

## 15. Biến thể của bài

1. **LeetCode 525 — Contiguous Array:** đoạn dài nhất có số 0 **bằng** số 1 ⇒ `P[i] = P[j]`, dùng dict lưu vị trí đầu tiên.
2. **LeetCode 560 — Subarray Sum Equals K:** đếm `P[j] − P[i] = k` ⇒ dict đếm.
3. **LeetCode 315 / 493 — Count of Smaller Numbers After Self / Reverse Pairs (Hard):** đếm cặp bằng BIT/merge sort.
4. **Đếm đoạn có `ones − zeros ≥ k`** (bài tập 5) ⇒ đếm `P[i] ≤ P[j] − k` bằng BIT.
5. **Đếm đoạn có số 1 nhiều hơn số 0 với độ dài tối thiểu `L`** ⇒ chỉ thêm `P[j − L]` vào cấu trúc khi tới `j`.

Minh họa biến thể 5 (tự dựng, kiểm bằng brute force):

In [29]:
def count_min_length(nums, L):
    n = len(nums); off = n + 1; bit = BIT(2 * n + 1)
    P = [0]
    for x in nums: P.append(P[-1] + (1 if x else -1))
    ans = 0
    for j in range(L, n + 1):
        bit.update(P[j - L] + off, 1)               # chỉ các i <= j - L được phép
        ans += bit.query(P[j] - 1 + off)
    return ans

def count_min_length_brute(nums, L):
    n = len(nums)
    return sum(1 for i in range(n) for j in range(i + L, n + 1)
               if 2 * sum(nums[i:j]) > j - i)

rng = random.Random(5)
for _ in range(300):
    a = [rng.randint(0, 1) for _ in range(rng.randint(1, 14))]; L = rng.randint(1, len(a))
    assert count_min_length(a, L) == count_min_length_brute(a, L)
print("Biến thể độ dài tối thiểu khớp brute force; ví dụ 1, L = 3 ->", count_min_length([0, 1, 1, 0, 1], 3))

Biến thể độ dài tối thiểu khớp brute force; ví dụ 1, L = 3 -> 5


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | mọi đoạn, cộng dồn | `O(n²)` | `O(1)` | kiểm tra |
| 2. Merge sort | đếm cặp khi trộn | `O(n log n)` | `O(n)` | khi quen đếm nghịch thế |
| 3. Fenwick | đếm tiền tố nhỏ hơn | `O(n log n)` | `O(n)` | tổng quát (bước bất kỳ) |
| 4. `bisect` + `insort` | list đã sắp | `O(n²)` worst, nhanh thực tế | `O(n)` | viết nhanh trong phỏng vấn |
| 5. Tuyến tính ±1 | cập nhật `less` | `O(n)` | `O(n)` | **nộp bài** |

## 17. Bài tập (từ dễ đến khó)

1. **Đoán output:** `[1, 0, 1]`.
2. **Tính tay:** `[1, 1, 0, 0, 1]` — lập bảng `s`, `less`, `ans` như mục 8.
3. **Điền TODO:** `prefix_balance(nums)` trả về mảng `P` (độ dài `n + 1`).
4. **Tự viết lại:** viết lại Cách 5 không nhìn code.
5. **Mini challenge:** `count_at_least_k(nums, k)` — số đoạn có `ones − zeros ≥ k` (`k ≥ 1`), `O(n log n)`. Kiểm bằng brute force.

In [30]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def prefix_balance(nums):
    P = [0]
    # TODO
    return P

In [31]:
# Bài 4 — TODO (chỉ định nghĩa, không gọi)
def my_count(nums):
    # TODO: cnt với offset, less, s
    pass

In [32]:
# Bài 5 — TODO (chỉ định nghĩa, không gọi)
def count_at_least_k(nums, k):
    # TODO: đếm cặp i < j với P[i] <= P[j] - k bằng BIT
    pass

## 18. Gợi ý 3 mức

**Bài 1.** (1) `P = [0, 1, 0, 1]`. (2) Cặp `P[i] < P[j]`: `(0,1)`, `(0,3)`, `(2,3)`. (3) Đáp án `3`.

**Bài 2.** (1) `P = [0, 1, 2, 1, 0, 1]`. (2) `less` lần lượt `1, 2, 1, 0, 2`. (3) Tổng `6`.

**Bài 3.** (1) Duyệt `x`. (2) `P.append(P[-1] + (1 if x else −1))`. (3) Độ dài `n + 1`.

**Bài 4.** (1) `cnt = [0]*(2n+1)`, `cnt[n] = 1`. (2) Gặp 1: cộng `cnt[s+n]` rồi tăng `s`. (3) Gặp 0: giảm `s` rồi trừ `cnt[s+n]`.

**Bài 5.** (1) `ones − zeros ≥ k` ⇔ `P[j] − P[i] ≥ k` ⇔ `P[i] ≤ P[j] − k`. (2) `query(P[j] − k + off)` (đếm `≤`). (3) Kẹp chỉ số về `[0, size]` khi `P[j] − k + off < 1`.

## 19. Đáp án tham khảo

In [33]:
# Bài 1, 2
assert Solution().subarraysWithMoreZerosThanOnes([1, 0, 1]) == 3
assert Solution().subarraysWithMoreZerosThanOnes([1, 1, 0, 0, 1]) == 6
print("Bài 1: 3 | Bài 2: 6")
trace_linear([1, 1, 0, 0, 1])

Bài 1: 3 | Bài 2: 6
j=1 x=1: less += cnt[0]=1, s -> 1     | less=1 ans=1 cnt={0: 1, 1: 1}
j=2 x=1: less += cnt[1]=1, s -> 2     | less=2 ans=3 cnt={0: 1, 1: 1, 2: 1}
j=3 x=0: s -> 1, less -= cnt[1]=1     | less=1 ans=4 cnt={0: 1, 1: 2, 2: 1}
j=4 x=0: s -> 0, less -= cnt[0]=1     | less=0 ans=4 cnt={0: 2, 1: 2, 2: 1}
j=5 x=1: less += cnt[0]=2, s -> 1     | less=2 ans=6 cnt={0: 2, 1: 3, 2: 1}


6

In [34]:
# Bài 3, 4
def prefix_balance_ans(nums):
    P = [0]
    for x in nums: P.append(P[-1] + (1 if x else -1))
    return P

my_count_ans = count_linear
assert prefix_balance_ans([0, 1, 1, 0, 1]) == [0, -1, 0, 1, 0, 1]
rng = random.Random(3)
for _ in range(300):
    a = [rng.randint(0, 1) for _ in range(rng.randint(1, 40))]
    P_ = prefix_balance_ans(a)
    assert my_count_ans(a) == sum(1 for i in range(len(P_)) for j in range(i + 1, len(P_)) if P_[i] < P_[j]) % MOD
print("Bài 3, 4: OK")

Bài 3, 4: OK


In [35]:
# Bài 5
def count_at_least_k_ans(nums, k):
    n = len(nums); off = n + 1; size = 2 * n + 1; bit = BIT(size)
    bit.update(off, 1); s = ans = 0
    for x in nums:
        s += 1 if x else -1
        idx = s - k + off
        if idx >= 1:
            ans += bit.query(min(idx, size))
        bit.update(s + off, 1)
    return ans % MOD

def count_at_least_k_brute(nums, k):
    n = len(nums); ans = 0
    for i in range(n):
        bal = 0
        for j in range(i, n):
            bal += 1 if nums[j] else -1
            ans += bal >= k
    return ans

rng = random.Random(5)
for _ in range(400):
    a = [rng.randint(0, 1) for _ in range(rng.randint(1, 30))]; k = rng.randint(1, 5)
    assert count_at_least_k_ans(a, k) == count_at_least_k_brute(a, k)
    assert count_at_least_k_ans(a, 1) == Solution().subarraysWithMoreZerosThanOnes(a)   # k = 1 là bài gốc
print("Bài 5: OK; ví dụ 1 với k = 2 ->", count_at_least_k_ans([0, 1, 1, 0, 1], 2))

Bài 5: OK; ví dụ 1 với k = 2 -> 2


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "đếm cặp tiền tố P[i] < P[j] với bước ±1":**

```python
cnt = [0] * (2 * n + 1); cnt[n] = 1     # offset n, P[0] = 0
s = less = ans = 0
for x in nums:
    if x == 1: less += cnt[s + n]; s += 1
    else:      s -= 1; less -= cnt[s + n]
    ans += less; cnt[s + n] += 1
return ans % MOD
```

In [36]:
def template(nums):
    n = len(nums); cnt = [0] * (2 * n + 1); cnt[n] = 1; s = less = ans = 0
    for x in nums:
        if x == 1: less += cnt[s + n]; s += 1
        else: s -= 1; less -= cnt[s + n]
        ans += less; cnt[s + n] += 1
    return ans % MOD
print([template(a) for a, _ in examples])

[9, 0, 1]


**Checklist:**

- [ ] Đổi `0 → −1`; đoạn hợp lệ ⇔ `P[i] < P[j]` (**nghiêm ngặt**).
- [ ] Có tiền tố rỗng `P[0] = 0`.
- [ ] Offset rõ ràng cho tiền tố âm (mảng đếm / BIT chỉ số ≥ 1).
- [ ] Cập nhật `less` đúng thứ tự: gặp 1 thì cộng trước rồi tăng `s`; gặp 0 thì giảm `s` rồi trừ.
- [ ] Lấy `% (10⁹ + 7)`.
- [ ] Đúng tên method `subarraysWithMoreZerosThanOnes` (theo doocs).